In [ ]:
from lets_plot import *

In [ ]:
import geopandas as gpd
import numpy as np
import spatialdata as sd
from spatialdata.models import ShapesModel
from spatialdata.transformations import get_transformation

import cellestial as cl

SHAPES = "Visium_HD_3prime_Mouse_Brain_cell_segmentations"
IMAGE = "Visium_HD_3prime_Mouse_Brain_lowres_image"
TABLE = "cell_segmentations"
COORDINATE_SYSTEM = "Visium_HD_3prime_Mouse_Brain_downscaled_lowres"

data = sd.read_zarr("data/spatialdata/data.zarr")

# v1 of cl.spatial only renders Point/circle shapes; this dataset stores
# cell segmentations as polygons, so swap them for their centroids.
poly = data.shapes[SHAPES]
data.shapes[SHAPES] = ShapesModel.parse(
    gpd.GeoDataFrame(
        {"radius": 5.0},
        geometry=poly.geometry.centroid,
        index=poly.index,
    ),
    transformations=get_transformation(poly, get_all=True),
)

table = data.tables[TABLE]
table.var_names_make_unique()
table.obs["total_counts"] = np.asarray(table.X.sum(axis=1)).ravel()

cl.spatial(
    data,
    key="total_counts",
    table_name=TABLE,
    image_name=IMAGE,
    shapes_name=SHAPES,
    coordinate_system=COORDINATE_SYSTEM,
    size=1,
)


In [ ]:
cl.retrieve(cl.spatial(
    data,
    key="region",
    table_name=TABLE,
    image_name=IMAGE,
    shapes_name=SHAPES,
    coordinate_system=COORDINATE_SYSTEM,
    size=0.8,
))

In [ ]:
data

In [ ]:
cl.spatial(
    data,
    key="total_counts",
    table_name="cell_segmentations",
    image_name="Visium_HD_3prime_Mouse_Brain_lowres_image",
    size=0.6,
) + scale_color_viridis()


In [ ]:
cl.spatial(
    data,
    key="total_counts",
    table_name="cell_segmentations",
    image_name="Visium_HD_3prime_Mouse_Brain_lowres_image",
)
